# Data processing

In [ ]:
import numpy as np 

import pandas as pd 

import matplotlib.pyplot as plt

import random

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader

In [ ]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
data = pd.read_csv('/kaggle/input/language-translation-englishfrench/eng_-french.csv', nrows=None)

In [ ]:
data = data.dropna()

In [ ]:
data.columns = ['en', 'fr']

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
from nltk import tokenize
import string
from unidecode import unidecode

In [ ]:
###sentence Tokenizing
data.map(lambda x: len(tokenize.sent_tokenize(x))).describe()

The vast majority of the dataset is only one sentence long.

In [ ]:
data_fr = '\n'.join(data['fr'].tolist())
chars_fr = sorted(list(set(data_fr)))


In [ ]:
data_en = '\n'.join(data['en'].tolist())
chars_en = sorted(list(set(data_en)))

In [ ]:
print(len(chars_en),len(chars_fr))

In [ ]:
print(len(data_fr),len(data_en))

In [ ]:
chars_en

In the lists of characters for English and French sentences we can see that we need to clean some of them.

In [ ]:
#Max features is the number of words, I did not set a limit for this notebook.
#The start, end, and unk token are important for the text processing and the fitting of the model.
max_features = None
start_token = '<START>'
end_token = '<END>'
unk_token = '<UNK>'
pad_token = '<PAD>'

In [ ]:
def clean_text(texts, star_token=start_token, end_token=end_token):
    texts = texts.lower() #convert uppercase to lowercase
    texts = unidecode(texts, errors='ignore') #convert accented letters into unaccented letters. Ignore unknown characters.
    texts = ''.join((char if char in (string.punctuation + string.ascii_lowercase) else ' ' for char in texts)) #keep the selected letters and punctuation.
    
    return texts

In [ ]:
data[['en_clean', 'fr_clean']] = data.map(clean_text)

Now we can look at the cleaned list of characters in the dataset.

In [ ]:
data_en = '\n'.join(data['en_clean'].tolist())
chars_en = sorted(list(set(data_en)))
chars_en

In [ ]:
data_fr = '\n'.join(data['fr_clean'].tolist())
chars_fr = sorted(list(set(data_fr)))
chars_fr

In [ ]:
sent_clean_fr = data['fr_clean'].tolist()
sent_clean_en = data['en_clean'].tolist()

We want to translate these sentences using a seq2seq model of tokenize words. So we need to tokenize the sentences of our dataset, build a dictionary from the tokenized words, and translate each sentences into a list of ids (each id corresponding to one word in the dictionary).

In [ ]:
from collections import Counter

I built a class for Tokenizing the sentences. You need to, first, fit the class to your dataset using fit(), by doing this a dictionnary will be build using the "max_tokens" most commom words. Second, you need to call the tokenizer to create a dataset of sentences with words tokens and ids tokens. <br>
Also, we add start_token at the start of each tokenized sentences, and end_token at their end (if a word is not inside the dictionnary of words, the unk_token is added).

In [ ]:
class Tokenize():
    def __init__(self, word_vocab=None, max_tokens=None, start_token='<START>', end_token='<END>', unk_token='<UNK>', pad_token='<PAD>'):
        self.max_tokens = max_tokens
        self.start_token = start_token
        self.end_token = end_token
        self.unk_token = unk_token
        self.pad_token = pad_token
        self.word_vocab = word_vocab
        if self.word_vocab:
            self.word_vocab = [self.pad_token, self.start_token, self.end_token, self.unk_token] + [word for word,count in word_count.most_common(max_tokens)]
            self.word_to_idx = {w:i for i,w in enumerate(self.word_vocab)}
            self.idx_to_word = {i:w for i,w in enumerate(self.word_vocab)}
            self.len_vocab = len(self.word_vocab)
    
    def fit(self, texts):
        
        if type(texts) == str: texts = [texts]
        
        word_count = Counter(tokenize.word_tokenize(' '.join(texts)))
        
        if self.max_tokens: 
            max_tokens = self.max_tokens -  3
        else:
            max_tokens = self.max_tokens
            
        self.word_vocab = [self.pad_token, self.start_token, self.end_token, self.unk_token] + [word for word,count in word_count.most_common(max_tokens)]
        self.word_to_idx = {w:i for i,w in enumerate(self.word_vocab)}
        self.idx_to_word = {i:w for i,w in enumerate(self.word_vocab)}
        self.len_vocab = len(self.word_vocab)
    
    def __call__(self, texts):
        if type(texts) == str: texts = [texts]
        texts_token = [[self.start_token] + tokenize.word_tokenize(text) + [self.end_token] for text in texts]
        texts_token_id = [[self.word_to_idx.get(word, self.word_to_idx[self.unk_token]) for word in sent] for sent in texts_token]
        
        return texts_token, texts_token_id
        

We split the dataset into train and validation sets. The tokenizers are fitted only on the train sets.

In [ ]:
from sklearn.model_selection import train_test_split

sent_clean_fr_train, sent_clean_fr_valid, sent_clean_en_train, sent_clean_en_valid = train_test_split(sent_clean_fr, sent_clean_en, test_size=0.2, random_state=0)

The french tokenizer.

In [ ]:
tokenize_fr = Tokenize()
tokenize_fr.fit(sent_clean_fr_train)
train_token_fr, train_id_fr = tokenize_fr(sent_clean_fr_train)
valid_token_fr, valid_id_fr = tokenize_fr(sent_clean_fr_valid)

The english tokenizer.

In [ ]:
tokenize_en = Tokenize()
tokenize_en.fit(sent_clean_en_train)
train_token_en, train_id_en = tokenize_en(sent_clean_en_train)
valid_token_en, valid_id_en = tokenize_en(sent_clean_en_valid)

In [ ]:
max_len_fr = max((len(text) for text in valid_token_fr))
max_len_fr

In [ ]:
max_len_en = max((len(text) for text in valid_token_en))
max_len_en

The longest sequence is in french with 54 token, so we put a maximum length of 64 for our model.

In [ ]:
max_len = 64

This function pad the sequences to the desired length. The pad token was defined previsously and set to have an id of 0. <br>
We are using teacher forcing for the english sequences therefore we need to shift the expected y values by one to the left. The model will take one word in the sequence and learn to predict the next word in the sequence until the end_token.

In [ ]:
def make_dataset(query, value, max_length=50, return_shift=False, pad_token_id=0):
        if len(query) > max_length: 
            query_pad = query[:max_length]
        else:
            query_pad = query + [pad_token_id]*(max_length-len(query))
            
        if len(value) > max_length:
            value_pad = value[:max_length]
            value_shifted_pad = value[1:max_length+1]
        else:
            value_pad = value[:-1] + [pad_token_id]*(max_length-len(value[:-1]))
            value_shifted_pad = value[1:] + [pad_token_id]*(max_length-len(value[1:]))
            
        
        return query_pad, value_pad, value_shifted_pad

In [ ]:
def shift(sent, pad_token_id=0):
    pad_token_shifted = sent[1:] + [pad_token_id] 
    return pad_token_shifted

In [ ]:
query_pad_train, value_pad_train, value_shifted_pad_train = map(np.array,zip(*map(make_dataset, train_id_fr, train_id_en, [max_len]*len(train_id_en))))
query_pad_valid, value_pad_valid, value_shifted_pad_valid = map(np.array,zip(*map(make_dataset, valid_id_fr, valid_id_en, [max_len]*len(train_id_en))))

An example of what the shift does.

In [ ]:
fr_vocab_size = tokenize_fr.len_vocab
en_vocab_size = tokenize_en.len_vocab

The number of words in each dictionary.

In [ ]:
print(fr_vocab_size, en_vocab_size)

In [ ]:
class CustomDataset(Dataset):
    def __init__(self,x,y):
        self.x=torch.tensor(x)
        self.y=torch.tensor(y)
    def __len__(self):
        return len(self.x)
    def __getitem__(self,idx):
        return self.x[idx],self.y[idx]

In [ ]:
train_dataset=CustomDataset(query_pad_train,value_pad_train)
val_dataset=CustomDataset(query_pad_valid,value_pad_valid)


In [ ]:
len(value_pad_train)

In [ ]:
batch_size=256

In [ ]:
train_loader=DataLoader(batch_size=batch_size,dataset=train_dataset,shuffle=True)
val_loader=DataLoader(batch_size=batch_size,dataset=val_dataset,shuffle=True)

# (Encoder--Decoder with Bahdanaou Attention)

In [ ]:
class Encoder(nn.Module):
  def __init__(self,vocab_size,hidden_sizes,dropout_rates,embed_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstms = nn.ModuleList()
    self.dropouts=nn.ModuleList()
    current_input_size=embed_dim
    for hidden_size,dropout_rate in zip(hidden_sizes,dropout_rates):
      self.lstms.append(nn.LSTM(current_input_size,hidden_size,batch_first=True))
      self.dropouts.append(nn.Dropout(dropout_rate))
      current_input_size=hidden_size
  def forward(self,x):
    x=self.embedding(x)
    all_hidden_states=[]
    all_cell_states=[]
    for lstm,dropout in zip(self.lstms,self.dropouts):
      x=dropout(x)
      x,(h,c)=lstm(x)
      all_hidden_states.append(h)
      all_cell_states.append(c)
    return x,all_hidden_states,all_cell_states

In [ ]:
class Attention(nn.Module):
  def __init__(self,encoder_hidden_dim,decoder_hidden_dim):
    super().__init__()
    self.Encoder_W=nn.Linear(encoder_hidden_dim,decoder_hidden_dim)
    self.Decoder_W=nn.Linear(decoder_hidden_dim,decoder_hidden_dim)
    self.proj=nn.Linear(decoder_hidden_dim,1,bias=False)
  
  def forward(self,top_encoder_outputs,top_decoder_output):
    top_decoder_output=top_decoder_output.squeeze(0)
    proj_decoder=self.Decoder_W(top_decoder_output.unsqueeze(1))
    proj_encoder=self.Encoder_W(top_encoder_outputs)

    attn_scores=self.proj(torch.tanh(proj_decoder+proj_encoder)).squeeze(2)       ### Make it [batch_size,seq_len]
    attn_weights=torch.nn.functional.softmax(attn_scores,dim=1)

    context=torch.bmm(attn_weights.unsqueeze(1),top_encoder_outputs)
    return context,attn_weights

In [ ]:
class Decoder(nn.Module):
  def __init__(self,vocab_size,decoder_hidden_sizes,encoder_hidden_sizes,dropout_rates,embed_dim):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim)
    self.lstms=nn.ModuleList()
    self.dropouts=nn.ModuleList()
    self.attention=Attention(encoder_hidden_sizes[-1],decoder_hidden_sizes[-1])
    current_input_size=embed_dim+encoder_hidden_sizes[-1]
    for hidden_size,dropout_rate in zip(decoder_hidden_sizes,dropout_rates):
      self.lstms.append(nn.LSTM(current_input_size,hidden_size,batch_first=True))
      self.dropouts.append(nn.Dropout(dropout_rate))
      current_input_size=hidden_size
    self.fc=nn.Linear(current_input_size+encoder_hidden_sizes[-1],vocab_size)
  def forward(self,x,decoder_hidden,decoder_cell,top_encoder_outputs):
    batch_size,seq_len=x.shape
    outputs=[]
    for t in range(seq_len):
      input=x[:,t].unsqueeze(1)
      input=self.embedding(input)
      context,weights=self.attention(top_encoder_outputs,decoder_hidden[-1])      ###Why was decoder hidden state only used in attn rather than both
      current_input=torch.cat((context,input),dim=2)      ### [batch_size,1,embed_dim+enc_dim] 

      all_hidden_states=[]
      all_cell_states=[]
      for i,(lstm,dropout) in enumerate(zip(self.lstms,self.dropouts)):
        current_input=dropout(current_input)
        current_input,(h,c)=lstm(current_input,(decoder_hidden[i],decoder_cell[i]))
        all_hidden_states.append(h)
        all_cell_states.append(c)
      decoder_hidden=all_hidden_states
      decoder_cell=all_cell_states
      pred=self.fc(torch.cat((current_input.squeeze(1), context.squeeze(1)), dim=1))
      outputs.append(pred.unsqueeze(1))
    predictions=torch.cat(outputs,dim=1)
    return predictions,decoder_hidden,decoder_cell

In [ ]:
class seq2seq_inference(nn.Module):
  def __init__(self,encoder,decoder,vocab_size):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.encoder.eval()
    self.decoder.eval()
    self.vocab_size=vocab_size
  def forward(self,source,device,sos_idx=tokenize_fr.word_to_idx['<START>'],eos_idx=tokenize_fr.word_to_idx['<END>'],max_len=50):
    source=source.to(device)
    batch_size=source.size(0)
    top_encoder_outputs,encoder_hidden,encoder_cell=self.encoder(source)
    decoder_hidden,decoder_cell=encoder_hidden,encoder_cell
    current_input_tok = torch.tensor([[sos_idx]], dtype=torch.long, device=device)
    outputs=[]
    for i in range(max_len):
      predictions,hidden,cell=self.decoder(current_input_tok,decoder_hidden,decoder_cell,top_encoder_outputs)
      predictions=predictions.squeeze(1)
      top_prediction=predictions.argmax(dim=1)
      current_input_tok=top_prediction.unsqueeze(1)
      outputs.append(top_prediction.item())
      if batch_size and top_prediction.item()==eos_idx:
        break
    return outputs

In [ ]:
class seq2seq_training(nn.Module):
  def __init__(self,encoder,decoder,vocab_size):
    super().__init__()
    self.encoder=encoder
    self.decoder=decoder
    self.vocab_size=vocab_size
  def forward(self,train_loader,val_loader,epochs,criterion,optimizer,device,eval_freq=100,eval_iter=5):
    global_step=0
    val_losses=[]
    total_losses=[]
    for i in range(epochs):
        total_loss=0
        for batch_src,batch_trg in train_loader:
            self.encoder.train()
            self.decoder.train()
            optimizer.zero_grad()
            batch_src,batch_trg=batch_src.to(device),batch_trg.to(device)
            top_encoder_outputs,encoder_hidden,encoder_cell=self.encoder(batch_src)

            decoder_hidden,decoder_cell=encoder_hidden,encoder_cell

            decoder_input,decoder_targets=batch_trg[:,:-1],batch_trg[:,1:]
            predictions,_,_=self.decoder(decoder_input,decoder_hidden,decoder_cell,top_encoder_outputs)

            loss=criterion(predictions.reshape(-1,predictions.shape[-1]),batch_trg[:,1:].reshape(-1))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.encoder.parameters(), max_norm=1.0)
            torch.nn.utils.clip_grad_norm_(self.decoder.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss+=loss.item()
            global_step+=1
            if global_step%(eval_freq)==0:
                self.encoder.eval()
                self.decoder.eval()
                total_val_loss=0
                with torch.no_grad():
                    for val_src,val_trg in val_loader:
                        val_src, val_trg = val_src.to(device), val_trg.to(device)
                        val_top,val_hidden,val_cell=self.encoder(val_src)
                        val_pred, _, _ = self.decoder(val_trg[:, :-1], val_hidden, val_cell, val_top)
                        val_loss = criterion(val_pred.reshape(-1, val_pred.shape[-1]), val_trg[:, 1:].reshape(-1))
                        total_val_loss+=val_loss.item()
                avg_val_loss=total_val_loss/len(val_loader)
                val_losses.append(val_loss)
                print(f"Ep {i+1} (step {global_step:06d}):" f"Val loss:{avg_val_loss:.3f}")
        total_losses.append(total_loss/len(train_loader)) 
        print(f"Epoch{i+1}: Training Loss:{total_loss/len(train_loader)}")
    return total_losses,val_losses

In [ ]:
embed_dim=300
epochs=50

In [ ]:
hidden_encoder_dim=[128,64]
hidden_decoder_dim=[128,64]
dropout_encoder=[0.4,0.4]
dropout_decoder=[0.4,0.4]

In [ ]:
encoder=Encoder(fr_vocab_size,hidden_encoder_dim,dropout_encoder,embed_dim)
decoder=Decoder(en_vocab_size,hidden_decoder_dim,hidden_encoder_dim,dropout_decoder,embed_dim)
train_model=seq2seq_training(encoder,decoder,en_vocab_size)
inference_model=seq2seq_inference(encoder,decoder,en_vocab_size)

In [ ]:
train_model
encoder.to(device)
decoder.to(device)

In [ ]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(lr=0.01,params=train_model.parameters())

In [ ]:
train_loss,val_loss=train_model(train_loader,val_loader,epochs,criterion,optimizer,device)

In [ ]:
ENCODER_SAVE_PATH = "encoder_weights.pt"
DECODER_SAVE_PATH = "decoder_weights.pt"
torch.save(encoder.state_dict(), ENCODER_SAVE_PATH)
torch.save(decoder.state_dict(), DECODER_SAVE_PATH)

In [ ]:
plt.plot(
        train_loss,
        label="train_loss"
    )

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.plot(
        val_loss,
        label="validation_loss"
    )

plt.xlabel("Steps")
plt.ylabel("Loss")
plt.legend()
plt.show()